# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **How many property sales in Pittsburgh above 500,000 last month?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-11-19 18:15:09 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-11-19T18:15:09.645370")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'property sales transactions Pittsburgh real estate':

1. **City-Owned Property (Real Estate Tax Database)** (53.5% match)
   Resource: City-Owned Properties
   Resource ID: `4ff5eb17-e2ad-4818-97c4-8f91fc6b6396`
   Dataset ID: `city-owned-property`
   Format: CSV | Rows: 5,108 | Cols: 19
   Tags: property_sales, real_estate, geographic_location, tax_records, sale_price, property_identification, municipal_districts, neighborhood_data, payment_history, treasury_sales
   This dataset contains real estate transaction information, including property ide
```


In [ ]:
# Step 1: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": 'property sales transactions Pittsburgh real estate', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `property sales`

**Result preview:**
```
Found 15 datasets matching 'property sales'

1. **Allegheny County Property Sale Transactions**
   ID: `real-estate-sales`
   This dataset contains data on all Real Property parcels that have sold since 2013 in Allegheny County, PA.  Before doing any market analysis on property sales, check the sales va
   - Property Sales Transactions (CSV) [DataStore] ID: `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
   - Allegheny County Information Portal Real Estate Sales (HTML) ID: `bfdf3a78-7830-425d-afa2-8fbc99c79dcb`
   - Parcels N'at (HTML) ID: `637ab4db-1313-4512-9c2b-6fa659759200`
   - 2012 Property Sal
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'property sales', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
**Limit:** 5

**Result preview:**
```
Resource: 5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1
Total records: 508,515
Loaded: 5
Fields (25): PARID, FULL_ADDRESS, PROPERTYHOUSENUM, PROPERTYFRACTION, PROPERTYADDRESSDIR, PROPERTYADDRESSSTREET, PROPERTYADDRESSSUF, PROPERTYADDRESSUNITDESC, PROPERTYUNITNO, PROPERTYCITY, PROPERTYSTATE, PROPERTYZIP, SCHOOLCODE, SCHOOLDESC, MUNICODE, MUNIDESC, RECORDDATE, SALEDATE, PRICE, DEEDBOOK, DEEDPAGE, SALECODE, SALEDESC, INSTRTYP, INSTRTYPDESC

Sample (5 rows):

           PARID                          FULL_ADDRESS  PROPERTYHOUSENUM PROPERTYFRACTION PROPERTYADDRESSDIR PROPERTYADDRESSSTREET PROPERTYADDRESSSUF
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 4: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT MAX("SALEDATE"::date) as max_date, MIN("SALEDATE"::date) as min_date FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != ''
```

**Result preview:**
```
SQL error (HTTP 500): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Internal Server Error", "message": "Internal Server Error"}, "success": false}
```


In [ ]:
# Step 4: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT MAX("SALEDATE"::date) as max_date, MIN("SALEDATE"::date) as min_date FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != \'\''
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 5: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT date_trunc('month', "SALEDATE"::date) as month, COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != '' AND "PROPERTYCITY" = 'PITTSBURGH' GROUP BY month ORDER BY month DESC LIMIT 12
```

**Result preview:**
```
SQL error (HTTP 500): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Internal Server Error", "message": "Internal Server Error"}, "success": false}
```


In [ ]:
# Step 5: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT date_trunc(\'month\', "SALEDATE"::date) as month, COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != \'\' AND "PROPERTYCITY" = \'PITTSBURGH\' GROUP BY month ORDER BY month DESC LIMIT 12'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 6: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT MAX("SALEDATE") as max_date, MIN("SALEDATE") as min_date FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != ''
```

**Result preview:**
```
SQL error (HTTP 500): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Internal Server Error", "message": "Internal Server Error"}, "success": false}
```


In [ ]:
# Step 6: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT MAX("SALEDATE") as max_date, MIN("SALEDATE") as min_date FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != \'\''
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 7: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT substring("SALEDATE",1,7) as ym, COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != '' AND "PROPERTYCITY" = 'PITTSBURGH' GROUP BY ym ORDER BY ym DESC LIMIT 12
```

**Result preview:**
```
SQL error (HTTP 409): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"query": ["(psycopg2.errors.UndefinedFunction) function substring(date, integer, integer) does not exist\nLINE 1: SELECT * FROM (SELECT substring(\"SALEDATE\", 1, 7) AS ym, cou...\n ^\nHINT: No function matches the given name and argument types. You might need to add explicit type casts.\n\n[SQL: SELECT *
```


In [ ]:
# Step 7: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT substring("SALEDATE",1,7) as ym, COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" IS NOT NULL AND "SALEDATE" != \'\' AND "PROPERTYCITY" = \'PITTSBURGH\' GROUP BY ym ORDER BY ym DESC LIMIT 12'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 8: SQL Analysis Query

**SQL:**
```sql
SELECT MAX("SALEDATE") as max_date FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1"
```

**Result preview:**
```
SQL: SELECT MAX("SALEDATE") as max_date FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" LIMIT 10000
Rows: 1
Columns: max_date

  max_date
2026-11-17
```


In [ ]:
# Step 8: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT MAX("SALEDATE") as max_date FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1"'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 9: SQL Analysis Query

**SQL:**
```sql
SELECT substring("SALEDATE"::text,1,7) as ym, COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' GROUP BY ym ORDER BY ym DESC LIMIT 15
```

**Result preview:**
```
SQL: SELECT substring("SALEDATE"::text,1,7) as ym, COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' GROUP BY ym ORDER BY ym DESC LIMIT 15
Rows: 15
Columns: ym, cnt

     ym  cnt
2026-11  132
2026-10 1141
2026-09 1241
2026-08 1379
2026-07 1603
2026-06 1586
2026-05 1429
2026-04 1328
2026-03 1288
2026-02  991
2026-01  979
2025-12 1527
2025-11 1321
2025-10 1487
2025-09 1425
```


In [ ]:
# Step 9: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT substring("SALEDATE"::text,1,7) as ym, COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' GROUP BY ym ORDER BY ym DESC LIMIT 15'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 10: SQL Analysis Query

**SQL:**
```sql
SELECT COUNT(*) as cnt, SUM(CAST("PRICE" AS FLOAT)) as total_value, AVG(CAST("PRICE" AS FLOAT)) as avg_value FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND substring("SALEDATE"::text,1,7) = '2026-10' AND CAST("PRICE" AS FLOAT) > 500000
```

**Result preview:**
```
SQL: SELECT COUNT(*) as cnt, SUM(CAST("PRICE" AS FLOAT)) as total_value, AVG(CAST("PRICE" AS FLOAT)) as avg_value FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND substring("SALEDATE"::text,1,7) = '2026-10' AND CAST("PRICE" AS FLOAT) > 500000 LIMIT 10000
Rows: 1
Columns: cnt, total_value, avg_value

 cnt  total_value  avg_value
 115  146379636.0  1272866.4
```


In [ ]:
# Step 10: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT COUNT(*) as cnt, SUM(CAST("PRICE" AS FLOAT)) as total_value, AVG(CAST("PRICE" AS FLOAT)) as avg_value FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' AND substring("SALEDATE"::text,1,7) = \'2026-10\' AND CAST("PRICE" AS FLOAT) > 500000'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 11: SQL Analysis Query

**SQL:**
```sql
SELECT COUNT(*) as total_sales_oct FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND substring("SALEDATE"::text,1,7) = '2026-10'
```

**Result preview:**
```
SQL: SELECT COUNT(*) as total_sales_oct FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND substring("SALEDATE"::text,1,7) = '2026-10' LIMIT 10000
Rows: 1
Columns: total_sales_oct

 total_sales_oct
            1141
```


In [ ]:
# Step 11: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT COUNT(*) as total_sales_oct FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' AND substring("SALEDATE"::text,1,7) = \'2026-10\''

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 12: SQL Analysis Query

**SQL:**
```sql
SELECT "SALECODE", "SALEDESC", COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND substring("SALEDATE"::text,1,7) = '2026-10' AND CAST("PRICE" AS FLOAT) > 500000 GROUP BY "SALECODE","SALEDESC" ORDER BY cnt DESC
```

**Result preview:**
```
SQL: SELECT "SALECODE", "SALEDESC", COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND substring("SALEDATE"::text,1,7) = '2026-10' AND CAST("PRICE" AS FLOAT) > 500000 GROUP BY "SALECODE","SALEDESC" ORDER BY cnt DESC LIMIT 10000
Rows: 8
Columns: SALECODE, SALEDESC, cnt

SALECODE                         SALEDESC  cnt
      AA                SALE NOT ANALYZED   74
       H                MULTI-PARCEL SALE   23
      99 CORRECTIVE DEED / DUPLICATE SALE    7
      35             CORPORATION TRANSFER    4
       0                       VALID SALE    3
```


In [ ]:
# Step 12: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "SALECODE", "SALEDESC", COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' AND substring("SALEDATE"::text,1,7) = \'2026-10\' AND CAST("PRICE" AS FLOAT) > 500000 GROUP BY "SALECODE","SALEDESC" ORDER BY cnt DESC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer (as delivered in chat)

> **## Property Sales Above $500,000 in Pittsburgh — Last Month

**115 property sales** in Pittsburgh (PROPERTYCITY = "PITTSBURGH") were recorded above $500,000 during the most recent complete month in the dataset, **October 2026** (data source: *Allegheny County Property Sale Transactions*, WPRDC, resource "Property Sales Transactions").

### Supporting numbers
- Total Pittsburgh sales in October 2026 (all price points): **1,141**
- Sales over $500,000: **115** (≈10% of all Pittsburgh sales that month)
- Combined sale value of those 115 transactions: **$146,379,636**
- Average sale price among them: **≈$1,272,866**

### Important data-quality caveat
The dataset itself warns that sale records must be screened for validity before market analysis — many recorded "sales" are not arm's-length market transactions. Breaking down the 115 sales >$500k by `SALECODE`:
- **74** — "Sale Not Analyzed" (not yet classified valid/invalid)
- **23** — "Multi-Parcel Sale" (price may reflect a bundle of parcels, not one property)
- **7** — "Corrective Deed / Duplicate Sale"
- **4** — "Corporation Transfer"
- **3** — "Valid Sale" (code 0 — the cleanest, confirmed arm's-length sales)
- **2** — "Exempt Buyer or Seller"
- **1** — "Bank/Financial Institution"
- **1** — "Preferential Assessment"

So while 115 is the raw count of sales records above $500,000, only 3 of them carry the "Valid Sale" designation; the majority (notably the 23 multi-parcel sales) may overstate true single-property values.

**Note on "last month":** The dataset's most recent month (November 2026) only has 132 partial-month records, so I used October 2026 as the most recent fully-populated month to represent "last month." If you'd like the analysis run for a different specific month, let me know and I can rerun it.**

*The figures above restate the assistant's answer for convenience — they are
not computed by this cell. The derivation lives in the code cells earlier in
this notebook; re-run them to re-derive every number from the source APIs.*

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Western PA Regional Data Center (WPRDC)
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-11-19

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-11-19 18:15:09
- **Query**: How many property sales in Pittsburgh above 500,000 last month?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
